# Agentic audio lab on Colab
Runtime → Change runtime type → **GPU**. Data and runs are kept on Google Drive, so a disconnect does not lose them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO = 'https://github.com/HanStaehlin/agentic-audio-lab.git'  # change if needed
WORK = '/content/drive/MyDrive/agentic-audio-lab'
!git clone -q $REPO /content/agentic-audio-lab || (cd /content/agentic-audio-lab && git pull -q)
%cd /content/agentic-audio-lab
!mkdir -p $WORK/data $WORK/runs $WORK/reports
!ln -sfn $WORK/data data && ln -sfn $WORK/runs runs && ln -sfn $WORK/reports reports
!pip install -q -r requirements.txt

## 1. Data (once)
VoiceBank-DEMAND from https://datashare.ed.ac.uk/handle/10283/2791. **Check the four download links on that page**; the URLs below follow its usual pattern but may need updating.

In [ ]:
import os
if not os.path.exists('data/vb16k/test/clean'):
    base = 'https://datashare.ed.ac.uk/bitstream/handle/10283/2791'
    for name in ['clean_trainset_28spk_wav', 'noisy_trainset_28spk_wav', 'clean_testset_wav', 'noisy_testset_wav']:
        !wget -q -O /content/{name}.zip {base}/{name}.zip
        !unzip -q -o /content/{name}.zip -d /content/raw
    !python scripts/prepare_data.py --raw-dir /content/raw --out-dir data/vb16k
!ls data/vb16k/train/clean | wc -l

## 2. Baseline (once)

In [ ]:
!python -m audiolab.train --config configs/baseline.json --run-dir runs/baseline
!cat runs/baseline/metrics.json

## 3. Agent
Add your Anthropic API key under the key icon (Secrets) as `ANTHROPIC_API_KEY`. Each launch asks for your approval below the cell.

In [ ]:
import os, sys
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

from agent.run_agent import main
sys.argv = ['run_agent', '--goal', 'Improve PESQ over the baseline while keeping rtf below 0.1. Use at most 6 runs.',
            '--baseline', 'baseline', '--max-runs', '6']
main()

In [ ]:
!ls -t reports | head -1 | xargs -I{} cat reports/{}